# 🔵 DBSCAN from Scratch
**Density-Based Spatial Clustering of Applications with Noise**

---

This notebook implements DBSCAN **entirely from scratch** — no scikit-learn, no scipy for the algorithm itself.

### What you'll learn
| Concept | Description |
|---|---|
| ε (epsilon) | Neighbourhood radius around each point |
| min_samples | Minimum points needed to form a dense region |
| Core point | Has ≥ min_samples neighbours within ε |
| Border point | Within ε of a core point, but not a core point itself |
| Noise point | Neither core nor border — labelled −1 |

### Notebook structure
1. Imports & constants  
2. Distance & neighbour functions  
3. Core DBSCAN algorithm  
4. Utilities (summary, classification)  
5. Visualisation helper  
6. Demo — Gaussian Blobs  
7. Demo — Two Moons (non-convex)  
8. Demo — Concentric Rings  
9. Parameter sensitivity analysis  
10. Comparison with scikit-learn

## 1 · Imports & Constants

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from collections import deque

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Sentinel labels used internally during the algorithm
UNVISITED = -2   # point not yet processed
NOISE     = -1   # point classified as noise

print('✅ Imports OK')

## 2 · Distance & Neighbour Search

DBSCAN's density estimation is built on two primitives:
- **Euclidean distance** between two points
- **ε-neighbourhood query** — find all points within radius ε

> 💡 The brute-force O(n²) neighbour search is simple and correct. For large datasets (n > 10k), replace it with a KD-tree (`scipy.spatial.cKDTree`) for O(n log n) queries.

In [ ]:
def euclidean_distance(p1: np.ndarray, p2: np.ndarray) -> float:
    """Euclidean distance between two points."""
    return np.sqrt(np.sum((p1 - p2) ** 2))


def get_neighbors(X: np.ndarray, point_idx: int, eps: float) -> list:
    """
    Return indices of all points within eps distance of X[point_idx].
    Brute-force O(n) — correct for any metric space.

    Parameters
    ----------
    X         : (n_samples, n_features) array
    point_idx : index of the query point
    eps       : neighbourhood radius ε

    Returns
    -------
    neighbors : list of int indices (includes point_idx itself)
    """
    neighbors = []
    for i, point in enumerate(X):
        if euclidean_distance(X[point_idx], point) <= eps:
            neighbors.append(i)
    return neighbors


# Quick sanity check
X_test = np.array([[0, 0], [0.1, 0.1], [5, 5]])
nb = get_neighbors(X_test, 0, eps=0.5)
print(f'Neighbours of point 0 within ε=0.5 → indices {nb}')  # expect [0, 1]

## 3 · DBSCAN Algorithm

The algorithm in plain English:

```
for each unvisited point p:
    N = neighbours(p, ε)
    if |N| < min_samples:
        mark p as NOISE
    else:
        start new cluster C
        add p to C
        seed_queue = N
        while seed_queue not empty:
            q = dequeue(seed_queue)
            if q was NOISE: add q to C (border point)
            if q unvisited:
                add q to C
                M = neighbours(q, ε)
                if |M| >= min_samples: enqueue M  ← q is a core point
```

In [ ]:
def dbscan(X: np.ndarray, eps: float = 0.5, min_samples: int = 5) -> np.ndarray:
    """
    DBSCAN clustering implemented from scratch.

    Parameters
    ----------
    X           : (n_samples, n_features) float array
    eps         : neighbourhood radius ε
    min_samples : min points (incl. self) to qualify as a core point

    Returns
    -------
    labels : (n_samples,) int array
        -1  → noise point
         0+ → cluster id
    """
    n = len(X)
    labels = np.full(n, UNVISITED, dtype=int)
    cluster_id = 0

    for i in range(n):
        if labels[i] != UNVISITED:
            continue  # already assigned — skip

        neighbors = get_neighbors(X, i, eps)

        # ── Not a core point → tentatively noise
        if len(neighbors) < min_samples:
            labels[i] = NOISE
            continue

        # ── Core point → expand new cluster via BFS
        labels[i] = cluster_id
        seed_queue = deque(neighbors)

        while seed_queue:
            j = seed_queue.popleft()

            # Former noise becomes a border point of this cluster
            if labels[j] == NOISE:
                labels[j] = cluster_id

            if labels[j] != UNVISITED:
                continue  # already claimed

            labels[j] = cluster_id
            j_neighbors = get_neighbors(X, j, eps)

            # j is a core point — propagate its neighbourhood
            if len(j_neighbors) >= min_samples:
                seed_queue.extend(j_neighbors)

        cluster_id += 1

    return labels


print('✅ dbscan() defined')

## 4 · Utilities

In [ ]:
def cluster_summary(labels: np.ndarray) -> dict:
    """Return a human-readable summary of clustering results."""
    unique = np.unique(labels)
    clusters = [c for c in unique if c != NOISE]
    noise_count = int(np.sum(labels == NOISE))
    return {
        'n_clusters'    : len(clusters),
        'n_noise'       : noise_count,
        'cluster_sizes' : {int(c): int(np.sum(labels == c)) for c in clusters},
    }


def point_types(X: np.ndarray, labels: np.ndarray,
                eps: float, min_samples: int) -> np.ndarray:
    """
    Classify each point as 'core', 'border', or 'noise'.

    Returns
    -------
    types : (n_samples,) array of strings
    """
    types = np.full(len(X), 'noise', dtype=object)
    for i in range(len(X)):
        if labels[i] == NOISE:
            continue
        nb = get_neighbors(X, i, eps)
        types[i] = 'core' if len(nb) >= min_samples else 'border'
    return types


def print_summary(summary: dict, dataset_name: str = '') -> None:
    """Pretty-print cluster summary."""
    header = f'Results — {dataset_name}' if dataset_name else 'Results'
    print(f'\n{'─'*40}')
    print(f'  {header}')
    print(f'{'─'*40}')
    print(f"  Clusters found : {summary['n_clusters']}")
    print(f"  Noise points   : {summary['n_noise']}")
    for cid, size in summary['cluster_sizes'].items():
        print(f'  Cluster {cid:<3}    : {size} points')
    print(f'{'─'*40}\n')


print('✅ Utilities defined')

## 5 · Visualisation Helper

In [ ]:
def plot_clusters(X: np.ndarray, labels: np.ndarray,
                  eps: float, min_samples: int,
                  title: str = 'DBSCAN Clustering',
                  show_types: bool = False) -> None:
    """
    2-D scatter plot coloured by cluster label.

    Parameters
    ----------
    X           : (n, 2) array of 2-D points
    labels      : cluster labels from dbscan()
    eps         : ε used (displayed in annotation)
    min_samples : min_samples used (displayed in annotation)
    title       : plot title
    show_types  : if True, outline core points with a larger ring
    """
    unique_labels = np.unique(labels)
    n_clusters = len([l for l in unique_labels if l != NOISE])
    palette = cm.get_cmap('tab10', max(n_clusters, 1))

    ptypes = point_types(X, labels, eps, min_samples) if show_types else None

    fig, ax = plt.subplots(figsize=(8, 6))

    for label in unique_labels:
        mask = labels == label
        if label == NOISE:
            ax.scatter(X[mask, 0], X[mask, 1],
                       c='black', marker='x', s=40, linewidths=1.2,
                       label='Noise', zorder=3)
        else:
            color = palette(label)
            ax.scatter(X[mask, 0], X[mask, 1],
                       color=color, s=50, edgecolors='white',
                       linewidths=0.5, label=f'Cluster {label}', zorder=2)
            # Highlight core points
            if show_types is not None and ptypes is not None:
                core_mask = mask & (ptypes == 'core')
                ax.scatter(X[core_mask, 0], X[core_mask, 1],
                           facecolors='none', edgecolors=color,
                           s=120, linewidths=1.5, zorder=4)

    summary = cluster_summary(labels)
    info = (f'ε={eps}  min_samples={min_samples}\n'
            f"Clusters: {summary['n_clusters']}  Noise: {summary['n_noise']}")
    ax.set_title(title, fontsize=14, pad=12)
    ax.text(0.02, 0.97, info, transform=ax.transAxes,
            fontsize=9, va='top', family='monospace',
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
    ax.legend(loc='lower right', fontsize=9, markerscale=1.2)
    ax.set_xlabel('x₁')
    ax.set_ylabel('x₂')
    plt.tight_layout()
    plt.show()


print('✅ plot_clusters() defined')

## 6 · Demo — Gaussian Blobs

Four well-separated Gaussian clusters with added noise. DBSCAN should recover all four cleanly.

In [ ]:
from sklearn.datasets import make_blobs

np.random.seed(42)
X_blobs, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.6, random_state=42)
X_blobs += np.random.normal(0, 0.15, X_blobs.shape)  # slight extra noise

EPS_BLOBS, MS_BLOBS = 0.8, 5

labels_blobs = dbscan(X_blobs, eps=EPS_BLOBS, min_samples=MS_BLOBS)
print_summary(cluster_summary(labels_blobs), 'Gaussian Blobs')

plot_clusters(X_blobs, labels_blobs,
              eps=EPS_BLOBS, min_samples=MS_BLOBS,
              title='DBSCAN — Gaussian Blobs',
              show_types=True)

## 7 · Demo — Two Moons (Non-Convex)

Two interleaving half-circles that k-means and Gaussian mixture models cannot separate. DBSCAN handles them naturally because it follows density, not shape.

In [ ]:
from sklearn.datasets import make_moons

X_moons, _ = make_moons(n_samples=300, noise=0.07, random_state=42)

EPS_MOONS, MS_MOONS = 0.15, 5

labels_moons = dbscan(X_moons, eps=EPS_MOONS, min_samples=MS_MOONS)
print_summary(cluster_summary(labels_moons), 'Two Moons')

plot_clusters(X_moons, labels_moons,
              eps=EPS_MOONS, min_samples=MS_MOONS,
              title='DBSCAN — Two Moons (non-convex)',
              show_types=True)

## 8 · Demo — Concentric Rings

Two rings of different radii — another shape that defeats centroid-based methods.

In [ ]:
from sklearn.datasets import make_circles

X_rings, _ = make_circles(n_samples=300, factor=0.4, noise=0.04, random_state=42)

EPS_RINGS, MS_RINGS = 0.12, 5

labels_rings = dbscan(X_rings, eps=EPS_RINGS, min_samples=MS_RINGS)
print_summary(cluster_summary(labels_rings), 'Concentric Rings')

plot_clusters(X_rings, labels_rings,
              eps=EPS_RINGS, min_samples=MS_RINGS,
              title='DBSCAN — Concentric Rings',
              show_types=True)

## 9 · Parameter Sensitivity Analysis

Vary ε and min_samples on the Two Moons dataset to see how each affects the result.

### 9a · Effect of ε

In [ ]:
eps_values = [0.05, 0.10, 0.15, 0.25, 0.40]

fig, axes = plt.subplots(1, len(eps_values), figsize=(18, 3.5))
palette = cm.get_cmap('tab10', 10)

for ax, eps in zip(axes, eps_values):
    lbl = dbscan(X_moons, eps=eps, min_samples=5)
    s = cluster_summary(lbl)
    unique = np.unique(lbl)
    for l in unique:
        mask = lbl == l
        if l == NOISE:
            ax.scatter(X_moons[mask, 0], X_moons[mask, 1],
                       c='black', marker='x', s=15, linewidths=0.8)
        else:
            ax.scatter(X_moons[mask, 0], X_moons[mask, 1],
                       color=palette(l), s=15, edgecolors='none')
    ax.set_title(f'ε={eps}\n{s["n_clusters"]} clusters, {s["n_noise"]} noise',
                 fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('Effect of ε  (min_samples=5, Two Moons)', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

### 9b · Effect of min_samples

In [ ]:
ms_values = [2, 5, 10, 20, 40]

fig, axes = plt.subplots(1, len(ms_values), figsize=(18, 3.5))

for ax, ms in zip(axes, ms_values):
    lbl = dbscan(X_moons, eps=0.15, min_samples=ms)
    s = cluster_summary(lbl)
    unique = np.unique(lbl)
    for l in unique:
        mask = lbl == l
        if l == NOISE:
            ax.scatter(X_moons[mask, 0], X_moons[mask, 1],
                       c='black', marker='x', s=15, linewidths=0.8)
        else:
            ax.scatter(X_moons[mask, 0], X_moons[mask, 1],
                       color=palette(l), s=15, edgecolors='none')
    ax.set_title(f'min_samples={ms}\n{s["n_clusters"]} clusters, {s["n_noise"]} noise',
                 fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('Effect of min_samples  (ε=0.15, Two Moons)', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

## 10 · Comparison with scikit-learn

Verify our scratch implementation against scikit-learn's optimised DBSCAN.

In [ ]:
from sklearn.cluster import DBSCAN as SklearnDBSCAN

EPS, MS = 0.15, 5

# Our implementation
our_labels  = dbscan(X_moons, eps=EPS, min_samples=MS)

# scikit-learn
sk_labels   = SklearnDBSCAN(eps=EPS, min_samples=MS).fit_predict(X_moons)

# Compare (cluster IDs may differ, but assignments should match)
our_s = cluster_summary(our_labels)
sk_s  = cluster_summary(sk_labels)

print('Metric              Our impl    sklearn')
print('─' * 42)
print(f'{"n_clusters":<20} {our_s["n_clusters"]:<12} {sk_s["n_clusters"]}')
print(f'{"n_noise":<20} {our_s["n_noise"]:<12} {sk_s["n_noise"]}')
for cid in our_s['cluster_sizes']:
    sk_size = sk_s['cluster_sizes'].get(cid, '—')
    print(f'{f"cluster {cid} size":<20} {our_s["cluster_sizes"][cid]:<12} {sk_size}')

# Plot side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, lbl, ttl in [
    (ax1, our_labels, 'Our DBSCAN (from scratch)'),
    (ax2, sk_labels,  'scikit-learn DBSCAN'),
]:
    for l in np.unique(lbl):
        mask = lbl == l
        if l == -1:
            ax.scatter(X_moons[mask, 0], X_moons[mask, 1],
                       c='black', marker='x', s=30, label='Noise')
        else:
            ax.scatter(X_moons[mask, 0], X_moons[mask, 1],
                       color=palette(l), s=30, label=f'Cluster {l}')
    ax.set_title(ttl, fontsize=12)
    ax.legend(fontsize=8)
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')

plt.suptitle(f'ε={EPS}, min_samples={MS}', fontsize=10)
plt.tight_layout()
plt.show()

---
## 🎯 Summary

| Property | DBSCAN |
|---|---|
| Number of clusters | Discovered automatically |
| Cluster shape | Arbitrary (density-connected) |
| Noise handling | Built-in (label = −1) |
| Key hyperparameters | ε (radius), min_samples |
| Time complexity | O(n²) brute-force, O(n log n) with KD-tree |
| Weakness | Struggles with varying-density clusters |

### Choosing ε
Plot a **k-distance graph** (sort distances to the k-th nearest neighbour). The "elbow" is a good ε.

### Next steps
- Replace brute-force neighbours with `scipy.spatial.cKDTree` for large datasets
- Try **HDBSCAN** (hierarchical DBSCAN) for varying-density clusters
- Implement **OPTICS** (Ordering Points To Identify the Clustering Structure)